# 03 — Knowledge-graph embedding controls

Phase 3 starts with the least glamorous model on purpose. This notebook embeds the **Iconclass taxonomy topology** with truncated SVD of an undirected adjacency matrix. It is a structural sanity control, not the final KGE model.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from caypollard.graphs.embeddings import adjacency_svd_embeddings
from caypollard.graphs.iconclass import child_edges, parse_notations
from caypollard.retrieval import top_k_cosine

In [ ]:
records = parse_notations(ROOT / "data/samples/iconclass_notations_fixture.txt")
edges = child_edges(records)
table = adjacency_svd_embeddings(edges, dimension=4, seed=42)
table.metadata

## Inspect a local graph neighbourhood

This is useful only as a pipeline check: a topology-derived representation should recover nearby taxonomy nodes. Evaluating it against the same hierarchy is partly circular and is therefore not treated as evidence for the multimodal claim.

In [ ]:
query = "25G411"
query_index = table.ids.index(query)
results = top_k_cosine(table.vectors[query_index], table.vectors, k=5, exclude_index=query_index)
[(table.ids[item.index], round(item.score, 4)) for item in results]

## Full hierarchy command

```bash
uv run python scripts/embed_iconclass_graph.py \
  data/raw/iconclass-core/notations.txt \
  results/embeddings/iconclass-adjacency-svd.npz \
  --dimension 64 --seed 42
```

The next graph models are relation-aware or random-walk baselines (Node2Vec/DeepWalk-style, RDF2Vec, ComplEx, RotatE). They must be compared against this deliberately simple control.

## Relation-aware graph controls

The taxonomy oracle is not enough. The next smoke test uses a tiny heritage-context graph with `part_of`, `created_by`, `printed_at`, and target `has_iconclass` edges. Node2Vec-style walks intentionally ignore predicate identity; RDF2Vec-style walks retain it.

In [ ]:
from caypollard.graphs.embeddings import node2vec_ppmi_embeddings, rdf2vec_ppmi_embeddings
from caypollard.graphs.triples import mask_target_relations, read_triples_tsv

triples = read_triples_tsv(ROOT / "data/samples/context_triples_fixture.tsv")
context_only = tuple(t for t in triples if t[1] != "has_iconclass")
node2vec = node2vec_ppmi_embeddings(
    context_only, dimension=4, walks_per_node=8, walk_length=8, window=2, seed=42
)
rdf2vec = rdf2vec_ppmi_embeddings(
    context_only, dimension=4, walks_per_entity=8, walk_length=6, window=2, seed=42
)
node2vec.metadata["relation_aware"], rdf2vec.metadata["relation_aware"]

## Target masking for G2

For a masked-label projection, evaluated image → Iconclass target edges are physically removed **before** fitting the graph model. The removed triples can be versioned and checksummed as an audit artifact.

In [ ]:
masked_graph, removed = mask_target_relations(
    triples, target_entities={"image:a", "image:c"}, relations={"has_iconclass"}
)
removed

ComplEx and RotatE use the optional PyKEEN backend (`uv sync --extra graph`). They are not executed in default CI because the repository keeps the lightweight research contract usable without pulling a second ML stack.